# Heart Disease Prediction - Model Training

This notebook builds a machine learning model that predicts whether a patient has heart disease based on a set of clinical measurements.

**Dataset:** [Heart Disease Dataset](https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset) (Kaggle, published by *johnsmith88*).

We are running this on Google Colab, so the dataset file downloaded from Kaggle has been uploaded to the Colab session's `/content` folder as `heart.csv`.

## 1. Load the dataset

We read `heart.csv` into a pandas DataFrame and take a first look at the rows and columns.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

DATA_PATH = "/content/heart.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

Shape: (1025, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


## 2. Quick look at the data

We check the data types, look for missing values, and see how balanced the target column is.

In [ ]:
df.info()
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Target counts (0 = no disease, 1 = disease):")
print(df["target"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1025 non-null   int64  
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   int64  
 11  ca        1025 non-null   int64  
 12  thal      1025 non-null   int64  
 13  target    1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB

Missing values per column:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target

In [ ]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,1025.0,54.434146,9.072290,29.0,48.0,56.0,61.0,77.0
sex,1025.0,0.695610,0.460373,0.0,0.0,1.0,1.0,1.0
cp,1025.0,0.942439,1.029641,0.0,0.0,1.0,2.0,3.0
trestbps,1025.0,131.611707,17.516718,94.0,120.0,130.0,140.0,200.0
chol,1025.0,246.000000,51.592510,126.0,211.0,240.0,275.0,564.0
fbs,1025.0,0.149268,0.356527,0.0,0.0,0.0,0.0,1.0
restecg,1025.0,0.529756,0.527878,0.0,0.0,1.0,1.0,2.0
thalach,1025.0,149.114146,23.005724,71.0,132.0,152.0,166.0,202.0
exang,1025.0,0.336585,0.472772,0.0,0.0,0.0,1.0,1.0
oldpeak,1025.0,1.071512,1.175053,0.0,0.0,0.8,1.8,6.2


## 3. Split features and target

The `target` column is what we want to predict. Everything else is an input feature.

We split the data into a training set and a testing set. The model learns on the training set and is evaluated on the testing set, which it has never seen before.

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows: ", X_test.shape[0])

Training rows: 820
Testing rows:  205


## 4. Build the model

We use a scikit-learn `Pipeline`. A pipeline keeps the preprocessing steps (filling missing values and scaling the numbers) bundled together with the classifier. This matters because the API later has to apply the *exact same* steps before predicting.

In [ ]:
model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42)),
])

model.fit(X_train, y_train)
print("Model trained.")

Model trained.


## 5. Evaluate on the test set

In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}\n")
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=["No disease", "Disease"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 1.0000

Classification report:
              precision    recall  f1-score   support

  No disease       1.00      1.00      1.00       100
     Disease       1.00      1.00      1.00       105

    accuracy                           1.00       205
   macro avg       1.00      1.00      1.00       205
weighted avg       1.00      1.00      1.00       205

Confusion matrix:
[[100   0]
 [  0 105]]


## 6. Save the model and its parameters

We save the trained pipeline as `model.joblib`. We also write the model settings and the test accuracy to `metrics.json` so the results are easy to report and to keep for reference.

In [ ]:
model_dir = Path("/content/model")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "model.joblib"
joblib.dump(model, model_path)

metrics = {
    "dataset": "johnsmith88/heart-disease-dataset",
    "model": "RandomForestClassifier",
    "n_estimators": 200,
    "test_size": 0.2,
    "random_state": 42,
    "accuracy": round(float(accuracy), 4),
    "features": list(X.columns),
}
(model_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))

print("Saved model to:", model_path)
print(json.dumps(metrics, indent=2))

Saved model to: /content/model/model.joblib
{
  "dataset": "johnsmith88/heart-disease-dataset",
  "model": "RandomForestClassifier",
  "n_estimators": 200,
  "test_size": 0.2,
  "random_state": 42,
  "accuracy": 1.0,
  "features": [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal"
  ]
}


## 7. Test the saved model

As a final check, we load `model.joblib` back from disk and make a prediction on a single patient. This is the same thing the FastAPI service will do when it receives a request.

In [ ]:
loaded_model = joblib.load(model_path)

sample = X_test.iloc[[0]]
prediction = int(loaded_model.predict(sample)[0])
probability = float(loaded_model.predict_proba(sample)[0][1])

print("Sample patient:")
print(sample.to_dict(orient="records")[0])
print()
print(f"Prediction: {prediction} ({'Disease' if prediction == 1 else 'No disease'})")
print(f"Probability of disease: {probability:.4f}")

Sample patient:
{'age': 64, 'sex': 1, 'cp': 0, 'trestbps': 120, 'chol': 246, 'fbs': 0, 'restecg': 0, 'thalach': 96, 'exang': 1, 'oldpeak': 2.2, 'slope': 0, 'ca': 1, 'thal': 2}

Prediction: 0 (No disease)
Probability of disease: 0.0200
